# Plotly for Finance - Interactive Data Visualization

In this notebook, you'll learn how to create interactive financial visualizations using Plotly and real market data from Yahoo Finance.

## Learning Objectives
- Fetch real financial data using yfinance
- Create interactive charts with Plotly Express and Graph Objects
- Build financial visualizations: line charts, candlesticks, scatter plots, and more
- Apply visualization techniques to portfolio analysis

## Setup and Installation

First, let's install and import the required libraries.

In [ ]:
# Install required packages (run once)
# uv add plotly

In [1]:
# Import libraries
import plotly.express as px #
import plotly.graph_objects as go #
from plotly.subplots import make_subplots #
import plotly.io as pio
pio.renderers.default = "notebook"
import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime, timedelta #

print("✓ All libraries imported successfully!")

✓ All libraries imported successfully!


---
## Part 1: Fetching Financial Data with yfinance

We'll use `yfinance` to download real stock market data.

In [13]:
tickers = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'NVDA']
end_date = datetime.now()
start_date = end_date - timedelta(days=365)

# Download data for all tickers
data = yf.download(tickers, start=start_date, end=end_date, group_by='ticker')

print(f"\n✓ Downloaded data for {len(tickers)} stocks")
print(f"✓ Date range: {len(data)} trading days")

C:\Users\fatih\AppData\Local\Temp\ipykernel_11700\2625159889.py:6: FutureWarning:

YF.download() has changed argument auto_adjust default to True

[*********************100%***********************]  5 of 5 completed


✓ Downloaded data for 5 stocks
✓ Date range: 250 trading days


In [3]:
# Let's examine the data structure
print("Data structure:")
print(data.head())
print("\nColumns:", data.columns.tolist()[:10], "...")

Data structure:
Ticker            AMZN                                                \
Price             Open        High         Low       Close    Volume   
Date                                                                   
2024-11-27  206.979996  207.639999  205.050003  205.740005  28061600   
2024-11-29  205.830002  208.199997  204.589996  207.889999  24892400   
2024-12-02  209.960007  212.990005  209.509995  210.710007  39523200   
2024-12-03  210.309998  214.020004  209.649994  213.440002  32214800   
2024-12-04  215.960007  220.000000  215.750000  218.160004  48745700   

Ticker            AAPL                                                ...  \
Price             Open        High         Low       Close    Volume  ...   
Date                                                                  ...   
2024-11-27  233.414318  234.628826  232.757286  233.872238  33498400  ...   
2024-11-29  233.752801  236.739294  232.916587  236.261459  28481400  ...   
2024-12-02  236.201715

---
## Part 2: Line Charts - Stock Price Trends

Line charts are essential for visualizing price movements over time.

In [12]:
# Prepare data for a single stock (Apple)
aapl_data = pd.DataFrame({
    'Date': data.index,
    'Close': data['AAPL']['Close'].values
})

# Create interactive line chart
fig = px.line(
    aapl_data,
    x='Date',
    y='Close',
    title='Apple (AAPL) Stock Price - Last Year',
    labels={'Close': 'Price (USD)', 'Date': 'Date'}
)

fig.update_layout(
    hovermode='x unified',
    template='plotly_white'
)

fig.show()

### Multiple Stock Comparison

Let's compare all stocks in our portfolio. To make comparisons fair, we'll normalize prices to show percentage changes.

In [6]:
# Create normalized price data (base = 100)
normalized_data = pd.DataFrame(index=data.index)

for ticker in tickers:
    prices = data[ticker]['Close']
    normalized_data[ticker] = prices

# Reshape data for plotly
plot_data = normalized_data.reset_index().melt(
    id_vars='Date',
    var_name='Stock',
    value_name='Normalized Price'
)

# Create multi-line chart
fig = px.line(
    plot_data,
    x='Date',
    y='Normalized Price',
    color='Stock',
    title='Tech Stocks Performance Comparison (Normalized to 100)',
    labels={'Normalized Price': 'Index (Base = 100)'}
)

fig.update_layout(
    hovermode='x unified',
    template='plotly_white',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1)
)

fig.show()

In [13]:
# Create normalized price data (base = 100)
normalized_data = pd.DataFrame(index=data.index)

for ticker in tickers:
    prices = data[ticker]['Close']
    normalized_data[ticker] = (prices / prices.iloc[0]) * 100

# Reshape data for plotly
plot_data = normalized_data.reset_index().melt(
    id_vars='Date',
    var_name='Stock',
    value_name='Normalized Price'
)

# Create multi-line chart
fig = px.line(
    plot_data,
    x='Date',
    y='Normalized Price',
    color='Stock',
    title='Tech Stocks Performance Comparison (Normalized to 100)',
    labels={'Normalized Price': 'Index (Base = 100)'}
)

fig.update_layout(
    hovermode='x unified',
    template='plotly_white',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1)
)

fig.show()

**Key Insight:** Normalized prices allow us to compare performance regardless of absolute price levels. A value of 120 means +20% return since the start.

---
## Part 3: Bar Charts - Volume Analysis

Trading volume is a key indicator of market activity and liquidity.

In [17]:
# Compare average daily volume across stocks
avg_volumes = pd.DataFrame({
    'Stock': tickers,
    'Avg Daily Volume': [data[ticker]['Volume'].mean() / 1_000_000 for ticker in tickers]  # in millions
})

fig = px.bar(
    avg_volumes,
    x='Stock',
    y='Avg Daily Volume',
    title='Average Daily Trading Volume Comparison',
    labels={'Avg Daily Volume': 'Volume (Millions)'},
    color='Avg Daily Volume',
    color_continuous_scale='Blues'
)

fig.update_layout(template='plotly_white')
fig.show()

---
## Part 4: Scatter Plots - Risk vs Return Analysis

One of the most important analyses in finance: understanding the relationship between risk (volatility) and return.

In [18]:
# Calculate returns and volatility for each stock
risk_return_data = []

for ticker in tickers:
    prices = data[ticker]['Close']
    returns = prices.pct_change().dropna()
    
    # Annualized return
    total_return = (prices.iloc[-1] / prices.iloc[0] - 1) * 100
    
    # Annualized volatility (risk)
    volatility = returns.std() * np.sqrt(252) * 100  # 252 trading days
    
    # Average volume
    avg_volume = data[ticker]['Volume'].mean()
    
    risk_return_data.append({
        'Stock': ticker,
        'Return (%)': total_return,
        'Volatility (%)': volatility,
        'Avg Volume': avg_volume
    })

risk_return_df = pd.DataFrame(risk_return_data)

# Create scatter plot
fig = px.scatter(
    risk_return_df,
    x='Volatility (%)',
    y='Return (%)',
    size='Avg Volume',
    color='Stock',
    title='Risk-Return Profile of Tech Stocks',
    hover_data=['Stock', 'Return (%)', 'Volatility (%)'],
    text='Stock'
)

# Add reference line at y=0
fig.add_hline(y=0, line_dash='dash', line_color='gray', annotation_text='Break-even')

fig.update_traces(textposition='top center')
fig.update_layout(template='plotly_white')
fig.show()

print("\nRisk-Return Summary:")
print(risk_return_df.to_string(index=False))


Risk-Return Summary:
Stock  Return (%)  Volatility (%)  Avg Volume
 AAPL   19.022599       32.742770  54779552.4
 MSFT   14.018147       24.453470  22036039.6
GOOGL   90.864852       33.337673  35810534.0
 AMZN   12.325640       35.145435  44479287.6
 NVDA   34.246696       49.809442 223674582.8


**Key Insight:** Stocks in the upper-left quadrant offer high returns with lower risk (ideal). Stocks in the lower-right have high risk but lower returns (less attractive).

---
## Part 5: Histograms - Return Distribution

Understanding the distribution of returns helps assess risk and probability of outcomes.

In [ ]:
# Calculate daily returns for Apple
aapl_returns = data['AAPL']['Close'].pct_change().dropna() * 100  # in percentage

fig = px.histogram(
    x=aapl_returns,
    nbins=50,
    title='Distribution of Daily Returns - Apple (AAPL)',
    labels={'x': 'Daily Return (%)', 'count': 'Frequency'}
)

# Add mean line
fig.add_vline(
    x=aapl_returns.mean(),
    line_dash='dash',
    line_color='red',
    annotation_text=f'Mean: {aapl_returns.mean():.2f}%',
    annotation_position='top right'
)

fig.update_layout(template='plotly_white', showlegend=False)
fig.show()

print(f"Return Statistics:")
print(f"Mean: {aapl_returns.mean():.3f}%")
print(f"Std Dev: {aapl_returns.std():.3f}%")
print(f"Min: {aapl_returns.min():.2f}%")
print(f"Max: {aapl_returns.max():.2f}%")

---
## Part 7: Subplots - Comprehensive Dashboard

Combine multiple charts to create a comprehensive analysis dashboard.

In [20]:
# Create subplots with 2 rows and 2 columns
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Price Trend',
        'Volume',
        'Daily Returns Distribution',
        'Cumulative Returns'
    ),
    specs=[
        [{'type': 'scatter'}, {'type': 'bar'}],
        [{'type': 'histogram'}, {'type': 'scatter'}]
    ]
)

# Get AAPL data
aapl = data['AAPL']
aapl_returns = aapl['Close'].pct_change().dropna()

# 1. Price trend
fig.add_trace(
    go.Scatter(x=aapl.index, y=aapl['Close'], name='Price', line=dict(color='blue')),
    row=1, col=1
)

# 2. Volume
fig.add_trace(
    go.Bar(x=aapl.index[-30:], y=aapl['Volume'][-30:]/1e6, name='Volume', marker_color='lightblue'),
    row=1, col=2
)

# 3. Returns distribution
fig.add_trace(
    go.Histogram(x=aapl_returns*100, name='Returns', nbinsx=30, marker_color='green'),
    row=2, col=1
)

# 4. Cumulative returns
cumulative_returns = (1 + aapl_returns).cumprod() - 1
fig.add_trace(
    go.Scatter(x=cumulative_returns.index, y=cumulative_returns*100, 
               name='Cumulative', line=dict(color='darkgreen')),
    row=2, col=2
)

# Update axes
fig.update_xaxes(title_text='Date', row=1, col=1)
fig.update_xaxes(title_text='Date', row=1, col=2)
fig.update_xaxes(title_text='Daily Return (%)', row=2, col=1)
fig.update_xaxes(title_text='Date', row=2, col=2)

fig.update_yaxes(title_text='Price (USD)', row=1, col=1)
fig.update_yaxes(title_text='Volume (M)', row=1, col=2)
fig.update_yaxes(title_text='Frequency', row=2, col=1)
fig.update_yaxes(title_text='Return (%)', row=2, col=2)

fig.update_layout(
    title_text='Apple (AAPL) - Comprehensive Dashboard',
    height=700,
    showlegend=False,
    template='plotly_white'
)

fig.show()

---
---
# 🎯 Exercises

Now it's your turn! Complete the following exercises to practice your Plotly skills.

## Exercise 1: Complete the Line Chart

**Goal:** Complete the code to create a line chart showing Microsoft's closing price.

**Instructions:**
1. Replace the `___` placeholders with the correct values
2. The chart should show MSFT closing prices over the entire period
3. Add proper labels and title

In [16]:
# tickers = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'NVDA']
# end_date = datetime.now()
# start_date = end_date - timedelta(days=365)

# # Download data for all tickers
# data = yf.download(tickers, start=start_date, end=end_date, group_by='ticker')

# data.head()

In [18]:
# Prepare data for Microsoft
msft_data = pd.DataFrame({
    'Date': data.index,
    'Close': data['MSFT']['Close'].values
})

# Create line chart
fig = px.line(
    msft_data,
    x='Date',  # TODO: Fill in x-axis column
    y='Close',  # TODO: Fill in y-axis column
    title='MSFT Stock Prices last 365 Days',  # TODO: Add a descriptive title
    labels={'Close': 'Close', 'Date': 'Date'}  # TODO: Add y-axis label
)

fig.update_layout(
    hovermode='x unified',
    template='plotly_white'
)

fig.show()

# TODO: Print the minimum and maximum price
print(f"Minimum price: ${msft_data['Close'].min():.2f}")
print(f"Maximum price: ${msft_data['Close'].max():.2f}")

Minimum price: $352.67
Maximum price: $541.06


---
## Exercise 2: Monthly Average Analysis

**Goal:** Calculate monthly average closing prices and create a bar chart.

**Instructions:**
1. Group the NVDA data by month
2. Calculate the average closing price for each month
3. Create a bar chart showing monthly averages
4. Add a trend line or annotation for the highest month

In [30]:
# Prepare NVDA data with proper index
nvda_prices = data['NVDA']['Close'].copy()

# TODO: Group by month and calculate mean
monthly_avg = nvda_prices.groupby(nvda_prices.index.month).mean()

# Create dataframe for plotting
monthly_df = pd.DataFrame({
    'Month': monthly_avg.index,
    'Average Price': monthly_avg.values
})

# TODO: Create bar chart
fig = px.bar(
    monthly_df,
    x='Month',  # TODO: Fill in x-axis
    y='Average Price',  # TODO: Fill in y-axis
    title='___',  # TODO: Add title
    labels={'Average Price': 'Avg Price (USD)', 'Month': 'Month'},
    color='Average Price',
    color_continuous_scale='Greens'
)

# TODO: Add annotation for the highest month
max_month = monthly_df.loc[monthly_df['Average Price'].idxmax()]
# Use fig.add_annotation() to highlight this point

fig.update_layout(template='plotly_white')
fig.show()

# TODO: Print summary statistics
print(f"Highest monthly average: ${max_month['Average Price']:.2f} in {max_month['Month']}")
print(f"Lowest monthly average: ${monthly_df['Average Price'].min():.2f}")
print(f"Overall monthly average: ${monthly_df['Average Price'].mean():.2f}")

Highest monthly average: $188.25 in 10.0
Lowest monthly average: $105.46
Overall monthly average: $149.28


---
## Exercise 3: Portfolio Allocation Pie Chart

**Goal:** Create a pie chart showing portfolio allocation based on current market values.

**Scenario:**
You own the following portfolio:
- AAPL: 50 shares
- MSFT: 40 shares
- GOOGL: 30 shares
- AMZN: 25 shares
- NVDA: 60 shares

**Instructions:**
1. Calculate the current value of each position (shares × current price)
2. Calculate the percentage allocation of each stock
3. Create an interactive pie chart
4. Add annotations showing both dollar value and percentage
5. Calculate and display total portfolio value

In [39]:
# Define portfolio holdings (number of shares)
portfolio = {
    'AAPL': 50,
    'MSFT': 40,
    'GOOGL': 30,
    'AMZN': 25,
    'NVDA': 60
}

# TODO: Get current (last) price for each stock
current_prices = {}
for ticker in portfolio.keys():
    current_prices[ticker] = data[ticker]['Close'].iloc[-1]

# TODO: Calculate current value of each position
position_values = {}
for ticker, shares in portfolio.items():
    position_values[ticker] = ___  # TODO: shares × price

# Calculate total portfolio value
total_value = sum(position_values.values())

# TODO: Calculate percentage allocation
allocations = {}
for ticker, value in position_values.items():
    allocations[ticker] = ___  # TODO: (value / total_value) × 100

# Create dataframe for plotting
portfolio_df = pd.DataFrame({
    'Stock': list(position_values.keys()),
    'Value': list(position_values.values()),
    'Allocation (%)': list(allocations.values())
})

# TODO: Create pie chart
fig = px.pie(
    portfolio_df,
    values='___',  # TODO: What should be the size of each slice?
    names='___',   # TODO: What should be the labels?
    title=f'Portfolio Allocation (Total Value: ${total_value:,.0f})',
    hole=0.3  # Creates a donut chart
)

# Customize hover template
fig.update_traces(
    textposition='inside',
    textinfo='percent+label',
    hovertemplate='<b>%{label}</b><br>Value: $%{value:,.0f}<br>Allocation: %{percent}<extra></extra>'
)

fig.update_layout(
    template='plotly_white',
    showlegend=True,
    legend=dict(orientation='v', yanchor='middle', y=0.5)
)

fig.show()

# TODO: Print detailed breakdown
print("\n📊 Portfolio Breakdown:")
print("="*60)
for ticker in portfolio.keys():
    print(f"{ticker:6} | {portfolio[ticker]:3} shares × ${current_prices[ticker]:7.2f} = ${position_values[ticker]:10,.2f} ({allocations[ticker]:5.2f}%)")
print("="*60)
print(f"{'TOTAL':6} | {'':>12} ${total_value:10,.2f} (100.00%)")

# TODO: Which stock has the highest allocation?
max_allocation_stock = portfolio_df.loc[portfolio_df['Allocation (%)'].idxmax(), 'Stock']
print(f"\n🏆 Highest allocation: {max_allocation_stock}")

ValueError: Value of 'names' is not the name of a column in 'data_frame'. Expected one of ['Stock', 'Value', 'Allocation (%)'] but received: ___

---
## Exercise 4: Multi-Stock Box Plot Analysis

**Goal:** Create box plots to compare daily return distributions across all stocks.

**Instructions:**
1. Calculate daily returns (percentage change) for all stocks
2. Prepare data in the correct format for box plots
3. Create a box plot showing return distributions for all 5 stocks
4. Add a reference line at y=0 (no return)
5. Color-code boxes by average return (positive = green, negative = red)
6. Calculate and display key statistics: median, IQR, and outliers

**Bonus Challenge:**
- Add violin plots to show the full distribution shape
- Identify and annotate the date with the biggest loss and gain for each stock

In [ ]:
# TODO: Calculate daily returns for all stocks
all_returns = pd.DataFrame()
for ticker in tickers:
    # Calculate percentage returns
    returns = data[ticker]['Close'].___  # TODO: Use pct_change()
    all_returns[ticker] = returns * 100  # Convert to percentage

# Remove NaN values
all_returns = all_returns.dropna()

# TODO: Reshape data for box plot (long format)
# Hint: Use .melt() to convert wide to long format
returns_long = all_returns.reset_index().melt(
    id_vars='___',  # TODO: What's the date column name?
    var_name='Stock',
    value_name='Daily Return (%)'
)

# TODO: Create box plot
fig = px.box(
    returns_long,
    x='___',  # TODO: Stock on x-axis
    y='___',  # TODO: Returns on y-axis
    color='Stock',
    title='Distribution of Daily Returns - Tech Stocks Comparison',
    labels={'Daily Return (%)': 'Daily Return (%)'}
)

# TODO: Add reference line at y=0
# Hint: Use fig.add_hline()
fig.___

fig.update_layout(
    template='plotly_white',
    showlegend=False
)

fig.show()

# TODO: Calculate and print statistics for each stock
print("\n📈 Return Statistics Summary:")
print("="*80)
print(f"{'Stock':6} | {'Median':>7} | {'Mean':>7} | {'Std Dev':>7} | {'Min':>8} | {'Max':>8}")
print("="*80)

for ticker in tickers:
    returns = all_returns[ticker]
    print(f"{ticker:6} | {returns.median():7.3f} | {returns.mean():7.3f} | {returns.std():7.3f} | {returns.min():8.2f} | {returns.max():8.2f}")

# TODO: Identify the most volatile stock
volatility = all_returns.std()
most_volatile = volatility.idxmax()
print(f"\n🔥 Most volatile stock: {most_volatile} (Std Dev: {volatility[most_volatile]:.3f}%)")

# TODO: Identify the stock with highest average return
avg_returns = all_returns.mean()
best_performer = avg_returns.idxmax()
print(f"🏆 Best performer: {best_performer} (Avg Daily Return: {avg_returns[best_performer]:.3f}%)")

### Bonus: Violin Plot for Exercise 4

If you completed Exercise 4, try creating a violin plot instead of a box plot. Violin plots show the full probability distribution.

In [ ]:
# Bonus: Violin plot (try this after completing Exercise 4)

# TODO: Create violin plot using the same data
fig = px.violin(
    returns_long,
    x='___',  # TODO: Fill in
    y='___',  # TODO: Fill in
    color='Stock',
    box=True,  # Show box plot inside violin
    points='outliers',  # Show outlier points
    title='Return Distribution - Violin Plot'
)

fig.add_hline(y=0, line_dash='dash', line_color='gray')
fig.update_layout(template='plotly_white', showlegend=False)
fig.show()

---
---
## 🎓 Resources
---
- [Plotly Documentation](https://plotly.com/python/)
- [yfinance Documentation](https://pypi.org/project/yfinance/)
- [Financial Data Analysis Guide](https://www.quantopian.com/lectures)